# Matthias: snow charts, map and statistical models

This notebook brings together the **25 November** exploration/map and **2 December** SQ1/SQ2 checkpoints. Run All uses saved inputs and makes no API downloads.

**Two evidence levels:** the real measurements cover 20 selected Swiss stations. The country comparison is a 24-area, four-country **unvalidated model demonstration**, not a validated result for every Alpine resort. We keep that distinction visible throughout.

A snow day means depth ≥30 cm between 1 December and 15 April. Winters are named by their ending year; periods are 1986–1995 and 2016–2025.


In [2]:
from pathlib import Path
import sys, html, sqlite3
import pandas as pd
from IPython.display import display, HTML

start = Path.cwd().resolve()
root = next(p for p in [start, *start.parents] if (p / "data/processed/meteoswiss_station_winters.csv").is_file())
sys.path.insert(0, str(root / "src"))
from snow_checkpoint_models import run_models
from snow_checkpoint_figures import make_figures
from snow_checkpoint_map import build_map
from checkpoint_database import build_crosswalk, update_checkpoint_database
from snow_checkpoint_report import write_report

results = run_models(root)
make_figures(root, results)
print("Computed the measured elevation regression, separate country-model demonstrations and station-level checks.")


## 1. What do the real measurements show?

The table uses one old/recent mean per station, then averages those station means. The graphical summary covers distributions, elevation groups, a regression and a past/recent comparison. Positive loss means fewer snow days recently.


In [3]:
measured = results["measured_data"]
table = measured.groupby("elevation_band", observed=True).agg(
    Stations=("station_id", "size"), Older_days=("observed_days_past", "mean"),
    Recent_days=("observed_days_recent", "mean"), Fewer_days=("observed_lost_days", "mean"))
display(table.round(2))
print(f"Across {len(measured)} selected Swiss stations: {measured.observed_lost_days.mean():.1f} fewer days per winter.")


,Stations,Older_days,Recent_days,Fewer_days
elevation_band,,,,
below_1500m,11,55.33,33.07,22.25
1500_to_below_2000m,8,102.76,94.89,7.87
at_least_2000m,1,136.10,136.30,-0.20


![Measured snow exploration](../docs/figures/checkpoint_measured_eda.png)

## 2. SQ1: the measured elevation regression

We estimate **measured loss = intercept + elevation association**. Elevation is centred at 1,500 m and measured in units of 100 m. There are 20 regression rows, not 400 independent locations. A country effect cannot be estimated from Swiss stations alone.


In [4]:
coefficients = results["coefficients"]
metrics = results["regression_metrics"]
display(coefficients.loc[coefficients.model_id.eq("sq1_measured_swiss_elevation")].round(4))
display(metrics.loc[metrics.model_id.eq("sq1_measured_swiss_elevation"),
    ["n_reference_sites", "r_squared", "adjusted_r_squared", "rmse_in_sample_days", "rmse_5fold_days", "rmse_5fold_training_mean_days"]].round(3))
slope = coefficients.loc[coefficients.model_id.eq("sq1_measured_swiss_elevation") & coefficients.term.eq("elevation_per_100m")].iloc[0]
print(f"In this selected Swiss sample, +100 m is associated with {abs(slope.estimate):.2f} fewer days of loss.")
print("Association only: site histories, sample selection and spatial dependence limit interpretation.")


,model_id,term,estimate,std_error,t_statistic,p_value_nominal,ci95_low,ci95_high
0,sq1_measured_swiss_elevation,Intercept,15.6741,2.5653,6.1101,0.0000,10.2846,21.0636
1,sq1_measured_swiss_elevation,elevation_per_100m,-1.6116,0.6499,-2.4796,0.0233,-2.9771,-0.2461


,n_reference_sites,r_squared,adjusted_r_squared,rmse_in_sample_days,rmse_5fold_days,rmse_5fold_training_mean_days
0,20,0.255,0.213,10.872,11.635,12.843


## 3. SQ1: the four-country model demonstration

This is a deliberately limited sample of six named ski areas per country. Coordinates were checked against source access pages before the snow outcomes were downloaded. The 480 seasons are reduced to one period difference per reference area. Duplicate returned model points are represented once in regressions.

We fit a shared elevation slope with country intercepts, then add **country × elevation interactions**. Switzerland is the reference; intercept contrasts refer to 1,500 m. These estimates describe model output whose agreement with real snow changes is poor.


In [5]:
model = results["model_data"]
display(model.groupby("country").agg(Areas=("resort_id", "size"), Base_min_m=("elev_base_m", "min"),
    Base_max_m=("elev_base_m", "max"), Older_model_days=("days_past", "mean"),
    Recent_model_days=("days_recent", "mean"), Model_loss=("lost_snow_days", "mean")).round(2))
display(pd.read_csv(root / "data/processed/snow_checkpoint_descriptive_statistics.csv").round(3))
display(pd.read_csv(root / "data/processed/snow_checkpoint_country_elevation_counts.csv"))


,Areas,Base_min_m,Base_max_m,Older_model_days,Recent_model_days,Model_loss
country,,,,,,
AT,6,620,1793,124.88,122.25,2.63
CH,6,665,2093,129.50,131.43,-1.93
FR,6,1000,1550,127.82,132.73,-4.92
IT,6,1162,2011,132.18,129.80,2.38


,source,group,variable,n,mean,median,std,min,q25,q75,max,skewness,excess_kurtosis
0,measured_swiss_stations,all,observed_days_past,20,78.340,81.35,35.071,14.4,55.175,106.975,136.1,-0.109,-1.038
1,measured_swiss_stations,all,observed_days_recent,20,62.960,64.10,40.687,9.1,29.425,94.200,136.3,0.340,-1.127
2,measured_swiss_stations,all,observed_lost_days,20,15.380,13.60,12.920,-4.3,9.200,25.800,47.1,0.511,0.420
3,measured_swiss_stations,below_1500m,observed_days_past,11,55.327,57.00,27.436,14.4,36.500,62.900,112.3,0.743,0.750
4,measured_swiss_stations,below_1500m,observed_days_recent,11,33.073,29.70,21.085,9.1,18.250,37.200,75.7,1.017,0.547
5,measured_swiss_stations,below_1500m,observed_lost_days,11,22.255,25.30,11.801,5.3,12.900,27.950,47.1,0.600,0.603
6,measured_swiss_stations,1500_to_below_2000m,observed_days_past,8,102.762,102.65,16.511,77.1,95.200,112.850,129.3,-0.044,-0.096
7,measured_swiss_stations,1500_to_below_2000m,observed_days_recent,8,94.888,95.10,22.827,63.0,82.575,107.350,130.4,0.021,-0.570
8,measured_swiss_stations,1500_to_below_2000m,observed_lost_days,8,7.875,11.30,8.990,-4.3,-1.400,14.475,18.4,-0.426,-1.916
9,measured_swiss_stations,at_least_2000m,observed_days_past,1,136.100,136.10,NaN,136.1,136.100,136.100,136.1,NaN,NaN


,country,below_1500m,1500_to_below_2000m,at_least_2000m
0,AT,3,3,0
1,CH,4,1,1
2,FR,5,1,0
3,IT,3,2,1


![Model demonstration exploration](../docs/figures/checkpoint_country_model_eda.png)

Country-specific lines stay within each country's sampled elevation range. Source-listed base elevation does not mean the grid's snow depth was corrected to that exact height.


In [6]:
display(coefficients.loc[coefficients.model_id.eq("sq1_model_country_elevation_interaction")].round(4))
display(metrics[["model_id", "n_reference_sites", "r_squared", "adjusted_r_squared",
    "rmse_in_sample_days", "rmse_5fold_days", "rmse_5fold_training_mean_days"]].round(3))
display(pd.read_csv(root / "data/processed/snow_checkpoint_anova.csv"))
display(pd.read_csv(root / "data/processed/snow_checkpoint_interaction_test.csv"))
print("Five-fold predictions train only on the other folds. Compare against the training-mean baseline; in-sample fit alone is not enough.")


,model_id,term,estimate,std_error,t_statistic,p_value_nominal,ci95_low,ci95_high
0,sq1_model_country_elevation_interaction,Intercept,-1.4834,1.6521,-0.8979,0.3826,-4.9858,2.0190
1,sq1_model_country_elevation_interaction,elevation_per_100m,0.2675,0.3221,0.8307,0.4184,-0.4152,0.9502
2,sq1_model_country_elevation_interaction,country_IT_vs_CH_at_1500m,3.9699,2.3123,1.7168,0.1053,-0.9320,8.8717
3,sq1_model_country_elevation_interaction,elevation_x_IT,-0.3802,0.5655,-0.6723,0.5110,-1.5790,0.8186
4,sq1_model_country_elevation_interaction,country_AT_vs_CH_at_1500m,3.4742,2.4070,1.4434,0.1682,-1.6284,8.5768
5,sq1_model_country_elevation_interaction,elevation_x_AT,-0.5625,0.4858,-1.1579,0.2639,-1.5924,0.4673
6,sq1_model_country_elevation_interaction,country_FR_vs_CH_at_1500m,1.6193,3.4474,0.4697,0.6449,-5.6889,8.9275
7,sq1_model_country_elevation_interaction,elevation_x_FR,1.4697,0.9477,1.5509,0.1405,-0.5392,3.4787


,model_id,n_reference_sites,r_squared,adjusted_r_squared,rmse_in_sample_days,rmse_5fold_days,rmse_5fold_training_mean_days
0,sq1_measured_swiss_elevation,20,0.255,0.213,10.872,11.635,12.843
1,sq1_model_country_elevation,24,0.439,0.321,3.579,4.404,5.106
2,sq1_model_country_elevation_interaction,24,0.573,0.387,3.122,4.945,5.106


,source,test,group,n_reference_sites,groups,status,statistic,p_value_nominal
0,measured_swiss_stations,one_way_anova,elevation_band,20,below_1500m:11;1500_to_below_2000m:8;at_least_...,insufficient_group_sizes,NaN,NaN
1,measured_swiss_stations,one_way_anova,elevation_group_1500m,20,at_least_1500m:9;below_1500m:11,exploratory; assumptions and spatial dependenc...,10.313099,0.004840
2,unvalidated_era5_land_sample,one_way_anova,country,24,AT:6;CH:6;FR:6;IT:6,exploratory; assumptions and spatial dependenc...,5.121028,0.008626
3,unvalidated_era5_land_sample,one_way_anova,elevation_band,24,below_1500m:15;1500_to_below_2000m:7;at_least_...,exploratory; assumptions and spatial dependenc...,1.319048,0.288650


,comparison,statistic,p_value_nominal,df_difference,source
0,country-specific elevation slopes versus share...,1.676735,0.211928,3.0,unvalidated four-country model demonstration


![Regression diagnostics](../docs/figures/checkpoint_regression_diagnostics.png)

## 4. SQ2: do modelled and measured changes agree?

The original comparison has 400 matched station winters. For change correlations and tests, we use **one summary per station (n=20)**. Pearson/Spearman measure association, not agreement. Nominal p-values and the station bootstrap do not remove geographic dependence.


In [7]:
display(pd.read_csv(root / "data/processed/snow_validation_metrics.csv").round(3))
display(results["tests"].round(5))
display(results["uncertainty"].round(3))
print("The model overcounts snow days and reproduces the measured period changes poorly. Raw model risk classes are not validated.")


,period,n_station_winters,n_stations,bias_days,mae_days,rmse_days,pearson_r,mean_daily_max_extra_days,model_full_season_fraction,observed_full_season_fraction
0,all,400,20,54.212,56.438,73.132,0.280,0.288,0.632,0.082
1,past,200,20,46.815,49.795,65.260,0.295,0.240,0.640,0.070
2,recent,200,20,61.610,63.080,80.235,0.272,0.335,0.625,0.095


,comparison,method,n_station_units,statistic,p_value_nominal,unit,inference_note
0,station_period_loss,Pearson,20,-0.18631,0.43159,one station,nominal; neighbouring stations may be dependent
1,station_period_loss,Spearman,20,-0.37411,0.10417,one station,nominal; neighbouring stations may be dependent
2,station_mean_snow_days,Pearson,20,0.34502,0.13627,one station,nominal; neighbouring stations may be dependent
3,station_mean_snow_days,Spearman,20,0.34913,0.13135,one station,nominal; neighbouring stations may be dependent
4,model_minus_observed_period_loss,paired_t_test,20,-4.77999,0.00013,one station,nominal; tests mean paired error in period los...


,metric,estimate_days,ci95_low,ci95_high,resampling_unit,bootstrap_draws,seed
0,mean_model_minus_observed_loss_days,-14.795,-20.815,-9.055,"whole station; exploratory, does not correct s...",5000,20261125
1,mean_model_minus_observed_snow_days,54.212,37.647,70.625,"whole station; exploratory, does not correct s...",5000,20261125


![Agreement in station changes](../docs/figures/checkpoint_station_change_tests.png)

## 5. Interactive map: ski areas and Italian closure records

The model sample is coloured by modelled loss. Gray points show the broader candidate roster, which still contains unreviewed geocodes. Closure locations are approximate municipality/locality positions, not individual lift GPS coordinates. Ambiguous matches remain listed and are not plotted.


In [8]:
map_result = build_map(root)
display(map_result["coverage"])
map_html = map_result["map_path"].read_text()
display(HTML('<iframe title="Ski areas and extracted Italian closures" style="width:100%;height:720px;border:1px solid #ccd5da" srcdoc="' + html.escape(map_html, quote=True) + '"></iframe>'))
print("Standalone map: docs/ski_resorts_closures_map.html (download and open in a browser).")


,layer,source_records,plotted_records
0,candidate_resorts,275,275
1,model_reference_sample,24,24
2,measured_swiss_stations,20,20
3,extracted_closure_records,184,181
4,unresolved_closures,3,0


![Static overview](../docs/figures/checkpoint_locations_map.png)

## 6. Fixing the pilot IDs and handing the data to Luca

The crosswalk preserves the two Zermatt/Cervinia gateways of one connected area. New model tables are explicitly labelled as checkpoint demonstrations. Existing team tables are preserved. These model results must not silently become validated snow-risk classes.


In [9]:
crosswalk = build_crosswalk(root)
join_audit = update_checkpoint_database(root, crosswalk)
display(crosswalk[["pilot_resort_id", "canonical_resort_id", "gateway_of_cross_border_area"]])
display(join_audit)
with sqlite3.connect(root / "db/sci_alpi.db") as connection:
    joined = pd.read_sql_query("""
        SELECT s.country, COUNT(*) AS n_reference_areas, AVG(s.lost_snow_days) AS mean_model_loss_days
        FROM checkpoint_snow_model_summary AS s
        JOIN resort_roster AS r ON s.resort_id = r.resort_id
        GROUP BY s.country ORDER BY s.country
    """, connection)
display(joined.round(3))
print("SQL output above describes unvalidated model estimates, not established country climate effects.")


,pilot_resort_id,canonical_resort_id,gateway_of_cross_border_area
0,zermatt-matterhorn,zermatt-breuil-cervinia-valtournenche-matterhorn,True
1,st-moritz-corviglia,st-moritz-corviglia,False
2,andermatt-sedrun-disentis,andermatt-oberalp-sedrun,False
3,breuil-cervinia-valtournenche-zermatt,zermatt-breuil-cervinia-valtournenche-matterhorn,True
4,livigno,livigno,False
5,aprica,aprica,False
6,kitzbuehel-kirchberg,kitzski-kitzbuehel-kirchberg,False
7,soelden,soelden,False
8,les-houches-saint-gervais-chamonix,les-houches-saint-gervais-prarion-bellevue-cha...,False
9,les-2-alpes,les-2-alpes,False


,join,input_rows,output_rows,lost_rows,distinct_canonical_areas,note
0,pilot entries to canonical area IDs,10,10,0,9,Zermatt and Cervinia remain two gateways of on...
1,model reference areas to canonical roster,24,24,0,24,one summary per source area; unvalidated snow ...


,country,n_reference_areas,mean_model_loss_days
0,AT,6,2.633
1,CH,6,-1.933
2,FR,6,-4.917
3,IT,6,2.383


## 7. What can I say to the group?

- We have measured snow loss and an elevation association for the selected Swiss stations.
- The model–station test is an important result: the model does not reproduce the measured changes well.
- The map, four-country model demonstration, regression diagnostics, ID crosswalk and database handover are implemented.
- We still need suitable validated snow data before making full-resort country conclusions or risk-based price/closure claims.

These results preserve the original study periods. A screen of another public station archive found no qualifying high-elevation Italian station spanning both target endpoints, so we did not quietly shorten the recent period. Direct Copernicus retrieval remains an account-dependent next data option.


In [10]:
write_report(root, results, map_result, join_audit)
print("Short group explanation: docs/matthias_group_update.md")
print("Detailed methods/results: docs/matthias_checkpoint_report.md")
print("All default steps completed using saved data; no API download needed.")
